In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [2]:
# diffraction grating animation
class Diffract:

    '''initialiser method'''
    def __init__(self, slits=100, d=1e-6, wavelength=6e-7, phi=0, x_range=4, y_range=4, resolution=200):

        # initial parameters
        self.slits_0 = slits
        self.d_0 = d
        self.wavelength_0 = wavelength
        self.phi_0 = phi
        self.x_range_0 = x_range
        self.y_range_0 = y_range
        self.resolution_0 = resolution

        self.anim_params = {}

        # initialise plot
        plt.style.use('dark_background')
        self.fig, self.ax = plt.subplots(figsize=(6,6))
        plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05)
        self.ax.get_xaxis().set_visible(False)
        self.ax.get_yaxis().set_visible(False)

    ''''method to allow paramter value variation'''
    def animate(self, name, start, end, endpoint=True):

        self.anim_params[name] = (start, end, endpoint)

    '''superpose all waves and build contour'''
    def __build_Z(self, slits, d, wavelength, positions, phi, x_range, y_range, resolution):

        # compute all points
        x = np.linspace(-x_range/2, x_range/2, resolution)
        y = np.linspace(0, y_range, resolution)
        self.X, self.Y = np.meshgrid(x, y)

        k = 2 * np.pi / wavelength
        
        Z = np.zeros_like(self.X)
        for pos in positions:
            Z += np.sin(k * np.sqrt((self.X - pos)**2 + self.Y**2) - phi)

        return Z

    '''build animation frame'''
    def __update(self, frame):

        # determine parameter value, whether animated or static
        slits = int(self.anim_slits[frame] if 'slits' in self.anim_params else self.slits_0)
        d = self.anim_d[frame] if 'd' in self.anim_params else self.d_0
        wavelength = self.anim_wavelength[frame] if 'wavelength' in self.anim_params else self.wavelength_0
        phi = self.anim_phi[frame] if 'phi' in self.anim_params else self.phi_0
        x_range = self.anim_x[frame] if 'x_range' in self.anim_params else self.x_range_0
        y_range = self.anim_y[frame] if 'y_range' in self.anim_params else self.y_range_0
        resolution = int(self.anim_res[frame] if 'resolution' in self.anim_params else self.resolution_0)

        # compute and plot contour for current frame
        grating_width = (slits - 1) * d
        positions = np.linspace(-grating_width/2, grating_width/2, slits)

        Z = self.__build_Z(slits, d, wavelength, positions, phi, x_range, y_range, resolution)

        self.ax.cla()
        self.ax.contourf(self.X, self.Y, Z, levels=200, cmap=self.cmap)

        # show white dots to represent slit positions
        if self.show_slits:
            x_vals = [pos for pos in positions if abs(pos) <= x_range/2]
            y_vals = np.zeros_like(x_vals)
            self.ax.scatter(x_vals, y_vals, s=15, color='white')

        # show values of current parameters
        if self.show_data:
            self.ax.set_title(
                f'slits={(slits):.3g}, d={d}, λ={(wavelength):.3g}, φ={(phi):.3g}, '
                f'x∈[{(-x_range/2):.3g}, {(x_range/2):.3g}], y∈[0, {(y_range):.3g}]'
            )

            try:
                if resolution == int(self.anim_res[frame]):
                    self.ax.text(
                        0.5, -0.04, f'resolution={resolution}×{resolution}', 
                        ha='center', va='bottom', transform=self.ax.transAxes
                    )
            except:
                pass

        return []

    '''display animation and save'''
    def show(
        self, frames=1, show_slits=False, show_data=False, 
        cmap='plasma', save=False, fps=2, title='DG', filetype='gif'
    ):

        # plot formatting parameters
        self.show_slits = show_slits
        self.show_data = show_data
        self.cmap = cmap

        # compute all variations of parameters for animation
        if 'slits' in self.anim_params:
            start, end, ep = self.anim_params['slits']
            self.anim_slits = np.linspace(start, end, frames, endpoint=ep)
            
        if 'd' in self.anim_params:
            start, end, ep = self.anim_params['d']
            self.anim_d = np.linspace(start, end, frames, endpoint=ep)
            
        if 'wavelength' in self.anim_params:
            start, end, ep = self.anim_params['wavelength']
            self.anim_wavelength = np.linspace(start, end, frames, endpoint=ep)
            
        if 'phi' in self.anim_params:
            start, end, ep = self.anim_params['phi']
            self.anim_phi = np.linspace(start, end, frames, endpoint=ep)
            
        if 'x_range' in self.anim_params:
            start, end, ep = self.anim_params['x_range']
            self.anim_x = np.linspace(start, end, frames, endpoint=ep)
            
        if 'y_range' in self.anim_params:
            start, end, ep = self.anim_params['y_range']
            self.anim_y = np.linspace(start, end, frames, endpoint=ep)

        if 'resolution' in self.anim_params:
            start, end, ep = self.anim_params['resolution']
            self.anim_res = np.linspace(start, end, frames, endpoint=ep)

        # build, save, and display animation
        if filetype == 'png':
            self.__update(0)
            if save:
                self.fig.savefig(f'{title}.png', dpi=300, bbox_inches='tight')
                
            return plt.show()

        if filetype == 'gif':
            anim = FuncAnimation(self.fig, self.__update, frames=frames, blit=True)
            if save:
                anim.save(f'{title}.gif', writer=PillowWriter(fps=fps))
            plt.close(self.fig)
            
            return HTML(anim.to_jshtml())

In [3]:
wave = Diffract(slits=20, d=1, wavelength=0.2, x_range=5, y_range=5)
wave.animate('phi', 0, 2*np.pi, False)
wave.show(frames=5, show_data=True, save=False, cmap='berlin_r')